In [0]:
%run ../00.common/1.environment_config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.sprints"
silver_table=f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df=spark.read.table(bronze_table)

In [0]:
sprints_required_df=sprints_df.drop("url")

In [0]:
sprint_renamed_df=(sprints_required_df
                   .withColumnsRenamed({"date":"race_date",
                                        "raceName":"race_name",
                                        "constructorId":"constructor_id",
                                        "driverId":"driver_id",
                                        "grid":"grid_position",
                                        "laps":"completed_laps",
                                        "number":"car_number",
                                        "position":"finish_position",
                                        "positionText":"finish_position_text"
                                    })
)


In [0]:
sprints_filtered_df=(sprint_renamed_df
                     .filter(F.col("season").isNotNull() &
                             F.col("round").isNotNull() &
                             F.col("constructor_id").isNotNull() &
                             F.col("driver_id").isNotNull() 
                     ))
                             

In [0]:
display((sprint_renamed_df.count())-(sprints_filtered_df.count()))

In [0]:
sprints_distinct_df=sprints_filtered_df.dropDuplicates(["season","round","constructor_id","driver_id"])


In [0]:
sprints_final_df=(sprints_distinct_df
                         .withColumn("race_name",F.initcap(F.col("race_name")))
)

In [0]:
(
    sprints_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)